In [1]:
# Cell 1: Install
!pip install -q kaggle gradio

# Cell 2: Check GPU
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))  # must show GPU

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# Cell 3
import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces --unzip -p /content/dataset

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
License(s): other
100% 3.75G/3.75G [00:38<00:00, 106MB/s]



In [3]:
import tensorflow as tf

IMG_SIZE = 224
BATCH = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    '/content/dataset/real_vs_fake/real-vs-fake/train',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    label_mode='binary'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    '/content/dataset/real_vs_fake/real-vs-fake/valid',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    label_mode='binary'
)

# Normalize
norm = tf.keras.layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (norm(x), y)).prefetch(tf.data.AUTOTUNE)
val_ds   = val_ds.map(lambda x, y: (norm(x), y)).prefetch(tf.data.AUTOTUNE)

print("Train:", train_ds)
print("Val:", val_ds)

Found 100000 files belonging to 2 classes.
Found 20000 files belonging to 2 classes.
Train: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 1), dtype=tf.float32, name=None))>
Val: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 1), dtype=tf.float32, name=None))>


In [ ]:
# Reset everything and start fresh with a simpler approach
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, Model
import tensorflow as tf

# Rebuild base with MobileNetV2 - faster and works better on this dataset
base = MobileNetV2(weights='imagenet', include_top=False,
                   input_shape=(224, 224, 3))

# Freeze ALL layers first
base.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

model = Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),  # higher LR this time
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train only on 20% of data to go faster
import tensorflow as tf
fast_train = train_ds.take(600)  # ~20k images
fast_val = val_ds.take(100)

history2 = model.fit(
    fast_train,
    validation_data=fast_val,
    epochs=5,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(patience=1, factor=0.5, verbose=1)
    ]
)

model.save('deepfake_mobilenet.keras')
print("✅ Done!")

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/5
599/600 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.7018 - loss: 0.6264

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

def show_gradcam_mobile(img_path):
    img = tf.keras.utils.load_img(img_path, target_size=(224, 224))
    arr = tf.keras.utils.img_to_array(img) / 255.0
    arr_expanded = np.expand_dims(arr, 0).astype('float32')

    # Build grad model using submodel's input/output directly
    mobilenet_submodel = model.get_layer('mobilenetv2_1.00_224')
    last_layer = mobilenet_submodel.get_layer('block_16_depthwise_relu')

    # Create a model from mobilenet input to last conv layer
    inner_model = tf.keras.Model(
        inputs=mobilenet_submodel.input,
        outputs=last_layer.output
    )

    with tf.GradientTape() as tape:
        conv_out = inner_model(arr_expanded, training=False)
        tape.watch(conv_out)
        # manually pass through rest of model
        x = model.get_layer('global_average_pooling2d_5')(conv_out)
        x = model.get_layer('dense_4')(x)
        x = model.get_layer('batch_normalization_1')(x)
        x = model.get_layer('dropout_4')(x, training=False)
        pred = model.get_layer('dense_5')(x)
        loss = pred[:, 0]

    grads = tape.gradient(loss, conv_out)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    cam = conv_out[0] @ pooled_grads[..., tf.newaxis]
    cam = tf.squeeze(cam).numpy()
    cam = np.maximum(cam, 0)
    cam = cv2.resize(cam, (224, 224))
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap_rgb = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(np.uint8(arr * 255), 0.6, heatmap_rgb, 0.4, 0)

    score = float(model.predict(arr_expanded, verbose=0)[0][0])
    label = "FAKE" if score > 0.5 else "REAL"
    color = 'red' if label == 'FAKE' else 'green'

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

In [ ]:
import gradio as gr
import numpy as np
from PIL import Image

def predict_deepfake(image):
    img = image.resize((224, 224))
    arr = np.array(img) / 255.0
    arr = np.expand_dims(arr, 0).astype('float32')
    score = float(model.predict(arr, verbose=0)[0][0])
    return {
        "FAKE 🔴": round(score, 3),
        "REAL 🟢": round(1 - score, 3)
    }

demo = gr.Interface(
    fn=predict_deepfake,
    inputs=gr.Image(type="pil", label="Upload a face image"),
    outputs=gr.Label(num_top_classes=2, label="Prediction"),
    title="🔍 Deepfake Face Detector",
    description="Upload any face image to check if it's AI-generated or real. Built with MobileNetV2 + Transfer Learning + Grad-CAM.",
    examples=[
        ['/content/dataset/real_vs_fake/real-vs-fake/test/fake/ZWEJOHNX2Z.jpg'],
        ['/content/dataset/real_vs_fake/real-vs-fake/test/real/52449.jpg']
    ]
)

demo.launch(share=True)

In [ ]:
# Use 50% of data instead of 20%
bigger_train = train_ds.take(1500)  # ~48k images
bigger_val = val_ds.take(200)

# Add data augmentation to make model more robust
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
])

# Apply augmentation to training data only
augmented_train = bigger_train.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

# Recompile with better settings
model.compile(
    optimizer=tf.keras.optimizers.Adam(5e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history3 = model.fit(
    augmented_train,
    validation_data=bigger_val,
    epochs=8,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.3, verbose=1)
    ]
)

model.save('deepfake_v2.keras')
print("✅ Done!")